# Day 3 — SQL for Analytics inside Colab
### DICT Data Analytics — Train the Trainer · Participant Notebook

**SQL for Analytics: SQLite in Colab, relational thinking, aggregation, and joins**

Aligned to the TESDA competencies *Prepare data sets* and *Summarize data sets* —
querying data with SELECT/WHERE/ORDER BY, grouping with GROUP BY/HAVING, and
joining tables with a validated row count.

---

**How to use this notebook**

1. Run the **Setup** cell first. Run it again any time the session restarts.
2. Work through Sections 1, 2 and 3. Each has a check cell at the end.
3. The checker tells you *what* went wrong, not just pass or fail. Read it
   before you ask for help.
4. Run **Final Check** when all three sections pass.

Work in pairs if possible. Explaining your reasoning to a partner is the closest thing to
teaching practice available inside a lab.

*Prepared by Nina Comia for the Department of Information and Communications
Technology.*

Setup — run this first

Colab clears files when the runtime disconnects. If variables vanish mid-session, re-run this cell; it is safe to run any number of times.

### Remember these 3 words throughout the day.

- **Table** = parang isang Excel sheet, may sariling pangalan (ex. `service_requests`)
- **Row** = isang linya ng datos (ex. isang `citizen request`)
- **Column** = isang detalye tungkol sa row na iyon (ex. `region`, `status`)

In [6]:
# ============================================================
#  SETUP  —  run this cell first, and run it again any time the session restarts
#  Creates the two course data files inside ./data/, and loads service_requests
#  into a SQLite database called dict_analytics.db as a table of the same name.
#    service_requests.csv   (1,222 rows, with deliberate defects)
#    regional_offices.json  (9 offices, nested structure)
# ============================================================
import pandas as pd, numpy as np, json, random, os, sqlite3

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)

np.random.seed(2026); random.seed(2026)
os.makedirs("data", exist_ok=True)

REGIONS = ["NCR", "Region III", "Region IV-A", "Region VI", "Region VII", "Region XI"]
OFFICES = {"NCR": ["DICT-NCR-01", "DICT-NCR-02"], "Region III": ["DICT-R3-01"],
           "Region IV-A": ["DICT-R4A-01", "DICT-R4A-02"], "Region VI": ["DICT-R6-01"],
           "Region VII": ["DICT-R7-01", "DICT-R7-02"], "Region XI": ["DICT-R11-01"]}
SERVICES = ["Free WiFi Installation", "Digital Literacy Training", "Business Permit Assistance",
            "ICT Equipment Request", "Cybersecurity Advisory", "eGov App Support"]
CHANNELS = ["Walk-in", "Online Portal", "Email", "Hotline", "Mobile App"]
STATUS = ["Resolved", "Pending", "Escalated", "Closed - No Action"]
AGE_GROUPS = ["18-24", "25-34", "35-44", "45-54", "55+"]


def build_dataset():
    """Build the Day 3 practice files. Synthetic, but it behaves like a real extract."""
    dates = pd.date_range("2025-01-01", "2025-12-31", freq="D")
    rows = []
    for i in range(1, 1201):
        region = random.choice(REGIONS)
        status = random.choices(STATUS, weights=[62, 20, 12, 6])[0]
        rows.append({
            "request_id": f"SR-2025-{i:05d}",
            "date_filed": random.choice(dates),
            "region": region,
            "office_code": random.choice(OFFICES[region]),
            "service_type": random.choice(SERVICES),
            "channel": random.choices(CHANNELS, weights=[30, 25, 15, 15, 15])[0],
            "status": status,
            "days_to_resolve": max(0, int(np.random.lognormal(1.6, 0.75))) if status in ("Resolved", "Closed - No Action") else np.nan,
            "satisfaction_rating": random.choice([1, 2, 3, 4, 5]) if status == "Resolved" else np.nan,
            "citizen_age_group": random.choice(AGE_GROUPS),
            "processing_fee": float(random.choice([0, 0, 0, 50, 100, 150, 200])),
        })
    df = pd.DataFrame(rows)

    # Inject realistic data-quality defects — the same ones a real government
    # export would have. You do not need to fix these today; you will meet them
    # again on Day 5. For now, just know they are there.
    d = df.copy()
    _ncr = d.index[d["region"] == "NCR"]
    d.loc[pd.Index(np.random.RandomState(1).choice(_ncr, 40, replace=False)), "region"] = " ncr "
    d.loc[d.sample(25, random_state=2).index, "service_type"] = "Free Wifi Installation"
    d.loc[d.sample(30, random_state=3).index, "citizen_age_group"] = np.nan
    d.loc[d.sample(18, random_state=4).index, "processing_fee"] = np.nan
    d.loc[d.sample(8, random_state=5).index, "days_to_resolve"] = 999
    d = pd.concat([d, d.sample(22, random_state=6)])
    d["date_filed"] = d["date_filed"].astype(str)
    _mix = d.sample(60, random_state=7).index
    d.loc[_mix, "date_filed"] = pd.to_datetime(d.loc[_mix, "date_filed"]).dt.strftime("%m/%d/%Y")
    d = d.sample(frac=1, random_state=8).reset_index(drop=True)
    return d


def build_offices():
    CITIES = {"NCR": "Quezon City", "Region III": "San Fernando", "Region IV-A": "Calamba",
              "Region VI": "Iloilo City", "Region VII": "Cebu City", "Region XI": "Davao City"}
    offices = []
    for region, codes in OFFICES.items():
        for c_ in codes:
            offices.append({
                "office_code": c_,
                "office_name": f"DICT Field Office {c_.split('-')[-1]} - {region}",
                "location": {"region": region, "city": CITIES[region],
                             "coordinates": {"lat": round(random.uniform(6.0, 16.5), 4),
                                             "lon": round(random.uniform(120.0, 126.0), 4)}},
                "staff_count": random.randint(8, 45),
                "year_established": random.randint(2012, 2022),
                "annual_budget_php": random.randrange(2_000_000, 12_000_000, 250_000),
                "services_offered": random.sample(SERVICES, k=random.randint(3, 6)),
                "is_active": random.random() > 0.1,
            })
    return offices


# Always regenerate BOTH files together. The offices data depends on the random
# state left behind by build_dataset(), so rebuilding one without the other would
# silently produce different numbers. Regenerating is instant, and it means this
# cell always restores a clean dataset if anything gets overwritten.
build_dataset().to_csv("data/service_requests.csv", index=False)
offices = build_offices()
with open("data/regional_offices.json", "w") as f:
    json.dump({"agency": "Department of Information and Communications Technology",
               "dataset": "Regional Field Office Master Data", "as_of": "2025-12-31",
               "record_count": len(offices), "offices": offices}, f, indent=2)

conn = sqlite3.connect("dict_analytics.db")
requests = pd.read_csv("data/service_requests.csv")
requests.to_sql("service_requests", conn, if_exists="replace", index=False)

with open("data/regional_offices.json") as f:
    _n_offices = json.load(f)["record_count"]

print("Setup complete. Files in ./data/:")
for fn in sorted(os.listdir("data")):
    print("   ", fn)
print()
print("CSV rows:", len(requests), "| Offices:", _n_offices)   # 1222 | 9
print("Loaded service_requests into dict_analytics.db")
print("Shape:", requests.shape)                               # (1222, 11)

Setup complete. Files in ./data/:
    regional_offices.json
    service_requests.csv

CSV rows: 1222 | Offices: 9
Loaded service_requests into dict_analytics.db
Shape: (1222, 11)


## Orientation — the four-query first look

Never query a database you have not looked at. Run these four before anything
else, every time, on every new table. Read the output rather than scrolling
past it.

In [7]:
# 1. What does a row actually look like?
pd.read_sql("""
SELECT *
FROM service_requests
LIMIT 5
""", conn)

,request_id,date_filed,region,office_code,service_type,channel,status,days_to_resolve,satisfaction_rating,citizen_age_group,processing_fee
0,SR-2025-01007,2025-12-17,Region III,DICT-R3-01,ICT Equipment Request,Hotline,Resolved,4.0,4.0,55+,0.0
1,SR-2025-00391,2025-03-01,Region III,DICT-R3-01,Cybersecurity Advisory,Walk-in,Pending,NaN,NaN,35-44,0.0
2,SR-2025-00794,2025-01-12,Region VI,DICT-R6-01,Free WiFi Installation,Online Portal,Pending,NaN,NaN,55+,50.0
3,SR-2025-00147,05/08/2025,Region XI,DICT-R11-01,Free WiFi Installation,Online Portal,Resolved,5.0,2.0,18-24,200.0
4,SR-2025-00387,2025-07-23,Region VII,DICT-R7-01,Cybersecurity Advisory,Hotline,Resolved,2.0,3.0,25-34,0.0


`pd.read_sql(...)` → sends the SQL query to the database and returns the result as a pandas DataFrame.

`SELECT *` means give me all the columns

`FROM service_requests` → get the data from the service_requests table.

`LIMIT 5` tells SQLite, to show only five records.

`conn` → tells pandas which database connection to use.

In [8]:
# 2. How many rows are we working with? Give it a shorter name: n
n = pd.read_sql("""
SELECT COUNT(*) AS n
FROM service_requests
""", conn)

n

,n
0,1222


`SELECT COUNT(*)` → counts all rows in the table including null.

`AS n` → gives the result a shorter column name or alias: n.

`FROM service_requests` → counts the rows in the service_requests table.

In [9]:
# 3. What columns and types does SQLite think this table has?
columns = pd.read_sql("""
PRAGMA table_info(service_requests)
""", conn)

columns[["name", "type"]]

,name,type
0,request_id,TEXT
1,date_filed,TEXT
2,region,TEXT
3,office_code,TEXT
4,service_type,TEXT
5,channel,TEXT
6,status,TEXT
7,days_to_resolve,REAL
8,satisfaction_rating,REAL
9,citizen_age_group,TEXT


In [10]:
# Then, from that information, show me only the column names and their data types.
columns[["name", "type"]]

,name,type
0,request_id,TEXT
1,date_filed,TEXT
2,region,TEXT
3,office_code,TEXT
4,service_type,TEXT
5,channel,TEXT
6,status,TEXT
7,days_to_resolve,REAL
8,satisfaction_rating,REAL
9,citizen_age_group,TEXT


This query asks SQLite for information about the columns in the service_requests table.

`"PRAGMA table_info(service_requests)"`returns information such as:

  cid → column position

  name → column name

  type → data type

  notnull → whether NULL is allowed

  dflt_value → default value

  pk → whether it's part of the primary key

In [11]:
# 4. What distinct values live in a categorical column status?
pd.read_sql("""
SELECT DISTINCT status
FROM service_requests
ORDER BY status
""", conn)

,status
0,Closed - No Action
1,Escalated
2,Pending
3,Resolved


In [12]:
# What distinct values live in a categorical column region?
pd.read_sql("""
SELECT DISTINCT region
FROM service_requests
ORDER BY region
""", conn)

,region
0,ncr
1,NCR
2,Region III
3,Region IV-A
4,Region VI
5,Region VII
6,Region XI


`DISTINCT` checks what unique values exist in the categorical column: status

In [13]:
# 5. A first real query — the five slowest Escalated requests
pd.read_sql("""
    SELECT request_id, region, service_type, days_to_resolve
    FROM   service_requests
    WHERE  status = 'Escalated'
    ORDER BY days_to_resolve DESC
    LIMIT  5
""", conn)

,request_id,region,service_type,days_to_resolve
0,SR-2025-00236,Region IV-A,Cybersecurity Advisory,999.0
1,SR-2025-00104,Region XI,ICT Equipment Request,NaN
2,SR-2025-00388,Region VII,Free WiFi Installation,NaN
3,SR-2025-00808,Region VI,Digital Literacy Training,NaN
4,SR-2025-00464,Region III,ICT Equipment Request,NaN


**Read as a sentence:**

“SELECT or get column/columns: request_id, region,
service_type, and days_to_resolve

FROM table service_requests

(Filter) WHERE status column is equal to Escalated,

(Sort) ORDER BY or arrange the results according to column days_to_resolve in DESC or from largest to smallest value.

LIMIT rows to 5.”

**Try this in every SQL query from now on**

In [14]:
pd.read_sql("SELECT request_id, region, service_type, days_to_resolve FROM service_requests", conn)

,request_id,region,service_type,days_to_resolve
0,SR-2025-01007,Region III,ICT Equipment Request,4.0
1,SR-2025-00391,Region III,Cybersecurity Advisory,NaN
2,SR-2025-00794,Region VI,Free WiFi Installation,NaN
3,SR-2025-00147,Region XI,Free WiFi Installation,5.0
4,SR-2025-00387,Region VII,Cybersecurity Advisory,2.0
...,...,...,...,...
1217,SR-2025-00137,Region IV-A,Cybersecurity Advisory,NaN
1218,SR-2025-00987,Region VII,Cybersecurity Advisory,7.0
1219,SR-2025-00134,Region VI,eGov App Support,2.0
1220,SR-2025-00362,Region VII,eGov App Support,NaN


In [15]:
pd.read_sql("SELECT request_id, region, service_type, days_to_resolve FROM service_requests WHERE status = 'Escalated' ORDER BY days_to_resolve DESC LIMIT 5", conn)

,request_id,region,service_type,days_to_resolve
0,SR-2025-00236,Region IV-A,Cybersecurity Advisory,999.0
1,SR-2025-00104,Region XI,ICT Equipment Request,NaN
2,SR-2025-00388,Region VII,Free WiFi Installation,NaN
3,SR-2025-00808,Region VI,Digital Literacy Training,NaN
4,SR-2025-00464,Region III,ICT Equipment Request,NaN


In [16]:
# 22 duplicated request_ids — the difference between 1222 and 1200
pd.read_sql("SELECT COUNT(*) AS total_rows, COUNT(DISTINCT request_id) AS unique_ids, "
  "COUNT(*) - COUNT(DISTINCT request_id) AS duplicate_rows FROM service_requests", conn)

,total_rows,unique_ids,duplicate_rows
0,1222,1200,22


## Checker

In [17]:
# ============================================================
#  CHECKER  —  run this once. You do not need to read it, but you may.
# ============================================================
RESULTS = {}


def _fmt(v):
    if isinstance(v, pd.DataFrame):
        return f"a DataFrame with {len(v)} rows"
    if isinstance(v, pd.Series):
        return f"a Series with {len(v)} values"
    return repr(v)


def _record(section, task, ok, got, want, hint, reveal=True):
    RESULTS[(section, task)] = ok
    mark = "PASS" if ok else "FAIL"
    print(f"  [{mark}]  Task {task}")
    if not ok:
        print(f"         you gave : {_fmt(got)}")
        if want is not None and reveal:
            print(f"         expected : {_fmt(want)}")
        if hint:
            print(f"         hint     : {hint}")


def _get(name):
    return globals().get(name, None)


def check_section_1():
    print("Section 1 \u2014 Relational thinking and basic queries")

    got = _get("total_rows")
    _record(1, 1, got == 1222, got, 1222,
            "SELECT COUNT(*) AS n FROM service_requests \u2014 then pull ['n'][0].")

    got = _get("pending_r7")
    ok = isinstance(got, pd.DataFrame) and len(got) == 40
    _record(1, 2, ok, got, "a DataFrame with 40 rows",
            "Two conditions joined with AND: status = 'Pending' and region = 'Region VII'.")

    got = _get("top_fees")
    ok = (isinstance(got, pd.DataFrame) and len(got) == 5
          and "processing_fee" in got.columns
          and (got["processing_fee"] == 200.0).all())
    _record(1, 3, ok, got, "5 rows, every processing_fee equal to 200.0",
            "ORDER BY processing_fee DESC, then LIMIT 5. Many rows are tied at 200 \u2014 that is expected.")

    got_pd = _get("pandas_rowcount")
    got_total = _get("total_rows")
    ok = got_pd is not None and got_total is not None and got_pd == got_total == 1222
    _record(1, 4, ok, got_pd, 1222,
            "pd.read_csv('data/service_requests.csv') into raw_df, then len(raw_df).")

    return _summary(1)


def check_section_2():
    print("Section 2 \u2014 Aggregation with GROUP BY and HAVING")

    got = _get("by_service")
    ok = (isinstance(got, pd.DataFrame) and len(got) == 7
          and got.iloc[0]["service_type"] == "Free WiFi Installation"
          and int(got.iloc[0]["request_count"]) == 213)
    _record(2, 1, ok, got, "7 rows, top row 'Free WiFi Installation' with count 213",
            "GROUP BY service_type, ORDER BY request_count DESC. Seven rows, not six \u2014 "
            "a mis-typed category ('Free Wifi Installation') forms its own group. That is a "
            "cleaning problem for Day 5, not a bug in your query.")

    got = _get("by_channel")
    ok = (isinstance(got, pd.DataFrame) and len(got) == 5
          and got.iloc[0]["channel"] == "Walk-in"
          and int(got.iloc[0]["resolved_requests"]) == 230)
    _record(2, 2, ok, got, "5 rows, top row 'Walk-in' with 230",
            "WHERE status = 'Resolved' BEFORE the GROUP BY.")

    got = _get("busy_offices")
    ok = (isinstance(got, pd.DataFrame) and len(got) == 3
          and set(got["office_code"]) == {"DICT-R6-01", "DICT-R3-01", "DICT-R11-01"})
    _record(2, 3, ok, got, "3 offices: DICT-R6-01, DICT-R3-01, DICT-R11-01",
            "HAVING COUNT(*) > 130, not WHERE.")

    got = _get("count_diff")
    _record(2, 4, got == 398, got, 398,
            "count_star (1222) minus count_days (824). These are the Pending and Escalated rows.")

    return _summary(2)


def check_section_3():
    print("Section 3 \u2014 Joins and validation")

    ok = True
    got = "offices table missing"
    try:
        n = pd.read_sql("SELECT COUNT(*) AS n FROM offices", conn)["n"][0]
        ok = n == 9
        got = f"{n} rows in offices"
    except Exception as e:
        ok = False
        got = f"query failed: {e}"
    _record(3, 1, ok, got, "9 rows in offices",
            "pd.json_normalize(payload['offices']), then .astype(str) on services_offered "
            "before .to_sql().")

    got = _get("joined")
    ok = isinstance(got, pd.DataFrame) and len(got) == 1222
    _record(3, 2, ok, got, "a DataFrame with 1222 rows",
            "INNER JOIN offices o ON r.office_code = o.office_code.")

    got = _get("join_rows")
    want_total = _get("total_rows")
    ok = got == 1222 and want_total == got
    _record(3, 3, ok, got, 1222,
            "len(joined), compared against total_rows from Section 1.")

    got = _get("workload")
    ok = False
    if isinstance(got, pd.DataFrame) and len(got) == 9 and "requests_per_staff" in got.columns:
        top = float(got["requests_per_staff"].max())
        # integer division truncates 12.25 -> 12, so check for the decimals,
        # not merely for a non-zero value.
        ok = abs(top - 12.25) < 0.01
    _record(3, 4, ok, got, "9 rows, highest requests_per_staff = 12.25",
            "If your top value is 12 rather than 12.25, SQLite did integer division "
            "and truncated the decimals \u2014 multiply the numerator by 1.0 before "
            "dividing. Note it does NOT return 0; it returns a plausible-looking "
            "whole number, which is what makes this bug dangerous.")

    return _summary(3)


def _summary(section):
    items = {k: v for k, v in RESULTS.items() if k[0] == section}
    passed = sum(items.values())
    total = len(items)
    print(f"\n  Section {section}: {passed} of {total} tasks passing.")
    if passed < total:
        print("  Read the hints above, fix the task, and run this cell again.")
    print()
    return passed == total


def check_everything():
    print("=" * 74)
    print("FINAL CHECK  \u2014  all three sections")
    print("=" * 74 + "\n")
    a = check_section_1()
    b = check_section_2()
    c = check_section_3()
    total = len(RESULTS)
    passed = sum(RESULTS.values())
    print("=" * 74)
    if a and b and c:
        print("""
   *  *  *   C O N G R A T U L A T I O N S   *  *  *

   All 12 tasks passed.

   You can now stand up a relational database inside a notebook, ask it
   direct questions with WHERE, GROUP BY and HAVING, and join two tables
   without silently duplicating a single row.

   More to the point, you caught a mis-typed category hiding inside
   'by_service', an office count that only appears once you group correctly,
   and a division bug that would have quietly shaved the decimals off every
   workload figure.

   That is the part worth teaching.

   Before you close this notebook:
     1. File > Download > .ipynb, and keep your copy.
     2. Write down one thing from today you would teach first, and why.
""")
        print("=" * 74)
    else:
        failed = [f"Section {s}, Task {t}" for (s, t), ok in sorted(RESULTS.items()) if not ok]
        print(f"\n   {passed} of {total} tasks passing. Still to fix:\n")
        for f in failed:
            print(f"     - {f}")
        print("\n   Run the section check cells above for the detail on each one.")
        print("=" * 74)


print("Checker loaded. Use check_section_1(), check_section_2(), check_section_3(),")
print("and check_everything() at the end.")

Checker loaded. Use check_section_1(), check_section_2(), check_section_3(),
and check_everything() at the end.


---
# Section 1 — Relational thinking and basic queries

**Hands-On 1 · 60 minutes**

SELECT, WHERE, ORDER BY, LIMIT — and the habit of checking your query against
a second source.

Write each number down on paper as you go, next to how confident you are it
is right, from one to five. You will need both after the break.

### Task 1.1 — Count every row

Set `total_rows` to the total number of rows in `service_requests`, using
`SELECT COUNT(*)`. Print it.

In [18]:
# Count the total number of rows in the service_requests table
# COUNT(*) counts every row in the table, including rows with NULL values.
total_rows = pd.read_sql("""
    SELECT COUNT(*) AS n
    FROM service_requests
""", conn)["n"].iloc[0]

# Print the total number of rows
print("total rows:", total_rows)

total rows: 1222


### Task 1.2 — Pending requests from Region VII

Set `pending_r7` to a DataFrame of `request_id`, `service_type` and `channel`
for rows where `status` is `'Pending'` **and** `region` is `'Region VII'`.

Remember: SQL string literals use single quotes, and string comparisons are
case-sensitive.

In [19]:
# Get pending requests from Region VII
# We only want the request_id, service_type, and channel columns.
pending_r7 = pd.read_sql("""
    SELECT request_id, service_type, channel
    FROM service_requests
    WHERE status = 'Pending'
      AND region = 'Region VII'
""", conn)

# Print the number of matching rows
print("rows:", len(pending_r7))

# Display the first 5 rows
pending_r7.head()


rows: 40


,request_id,service_type,channel
0,SR-2025-00086,Digital Literacy Training,Walk-in
1,SR-2025-01137,ICT Equipment Request,Email
2,SR-2025-01182,Cybersecurity Advisory,Mobile App
3,SR-2025-00392,eGov App Support,Walk-in
4,SR-2025-00752,Digital Literacy Training,Online Portal


### Task 1.3 — The five highest processing fees

Set `top_fees` to the 5 rows with the highest `processing_fee`, showing
`request_id`, `region` and `processing_fee`.

In [20]:
# Get the 5 requests with the highest processing fees
# Show only request_id, region, and processing_fee.
top_fees = pd.read_sql("""
    SELECT request_id, region, processing_fee
    FROM service_requests
    ORDER BY processing_fee DESC
    LIMIT 5
""", conn)

# Display the results
top_fees

,request_id,region,processing_fee
0,SR-2025-00147,Region XI,200.0
1,SR-2025-00388,Region VII,200.0
2,SR-2025-00799,Region III,200.0
3,SR-2025-00393,NCR,200.0
4,SR-2025-01028,ncr,200.0


### Task 1.4 — Cross-check against pandas

Read `data/service_requests.csv` directly with `pd.read_csv` into a variable
called `raw_df`, independent of the SQL table above. Set `pandas_rowcount` to
`len(raw_df)`.

You are about to compare this number to `total_rows`. They should match —
that comparison is the whole point of the task.

In [21]:
# Read the CSV file directly into a Pandas DataFrame
# This is independent of the SQL table.
raw_df = pd.read_csv("data/service_requests.csv")

# Count the total number of rows in the CSV
pandas_rowcount = len(raw_df)

# Print the Pandas row count
print("pandas rowcount  :", pandas_rowcount)

# Optional: compare it with the SQL row count
print("SQL rowcount     :", total_rows)

pandas rowcount  : 1222
SQL rowcount     : 1222


#### Check Section 1

In [22]:
check_section_1()

Section 1 — Relational thinking and basic queries
  [PASS]  Task 1
  [PASS]  Task 2
  [PASS]  Task 3
  [PASS]  Task 4

  Section 1: 4 of 4 tasks passing.



np.True_

**Analysis and Discussion 1**

- SQL and pandas returned the same row count. What does that tell you about
  where data quality problems actually live?
- Read Task 1.2's query aloud as an English sentence. Does the SQL order
  match the way you said it?
- When would you reach for SQL to pull a subset, instead of loading the
  whole table into pandas first?

---
# Section 2 — Aggregation with GROUP BY and HAVING

**Hands-On 2 · 60 minutes**

Collapse rows into summary numbers — and stop confusing WHERE with HAVING.

Order of writing: SELECT, FROM, WHERE, GROUP BY, HAVING, ORDER BY. Order of
execution is different: FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY.
WHERE runs before grouping and cannot see an aggregate like `COUNT(*)`;
HAVING runs after grouping and can.

In [23]:
pd.read_sql("""
SELECT region,
       COUNT(*)                        AS total_requests,
       COUNT(days_to_resolve)          AS resolved_count,
       ROUND(AVG(days_to_resolve), 2)  AS avg_days
FROM   service_requests
GROUP BY region
ORDER BY total_requests DESC
""", conn)

,region,total_requests,resolved_count,avg_days
0,Region IV-A,222,154,14.14
1,Region VI,218,137,13.42
2,Region III,208,146,26.87
3,Region XI,198,134,21.23
4,Region VII,179,117,14.13
5,NCR,157,111,6.19
6,ncr,40,25,5.92


In [24]:
# INTENTIONAL ERROR — run this in front of the room.
try:
    pd.read_sql("""
        SELECT office_code, COUNT(*) AS n
        FROM   service_requests
        WHERE  COUNT(*) > 130
        GROUP BY office_code
    """, conn)
except Exception as e:
    print("ERROR:", e)
# → misuse of aggregate function COUNT()

ERROR: Execution failed on sql '
        SELECT office_code, COUNT(*) AS n
        FROM   service_requests
        WHERE  COUNT(*) > 130
        GROUP BY office_code
    ': misuse of aggregate: COUNT()


### Task 2.1 — Requests and average resolution time per service type

Set `by_service` to `service_type`, a `request_count`, and the average
`days_to_resolve` rounded to 2 decimals, grouped by `service_type` and
ordered by count descending.

In [25]:
# Group the requests by service type
# Count the number of requests for each service type
# Calculate the average resolution time and round it to 2 decimal places
by_service = pd.read_sql("""
    SELECT
        service_type,
        COUNT(*) AS request_count,
        ROUND(AVG(days_to_resolve), 2) AS avg_days_to_resolve
    FROM service_requests
    GROUP BY service_type
    ORDER BY request_count DESC
""", conn)

# Display the result
by_service

,service_type,request_count,avg_days_to_resolve
0,Free WiFi Installation,213,26.77
1,eGov App Support,205,6.15
2,Business Permit Assistance,205,13.54
3,Digital Literacy Training,198,13.91
4,ICT Equipment Request,197,14.25
5,Cybersecurity Advisory,179,23.72
6,Free Wifi Installation,25,8.43


### Task 2.2 — Resolved requests per channel

Set `by_channel` to a count per `channel`, but only counting rows where
`status` is `'Resolved'`. This means the filter runs **before** the grouping
— use WHERE, not HAVING.

In [26]:
# Count resolved requests for each channel
# WHERE filters the rows BEFORE grouping.
by_channel = pd.read_sql("""
    SELECT
        channel,
        COUNT(*) AS resolved_requests
    FROM service_requests
    WHERE status = 'Resolved'
    GROUP BY channel
    ORDER BY resolved_requests DESC
""", conn)

# Display the results
by_channel

,channel,resolved_requests
0,Walk-in,230
1,Online Portal,176
2,Mobile App,123
3,Email,118
4,Hotline,108


### Task 2.3 — Offices with more than 130 requests

Set `busy_offices` to `office_code` and a `request_count`, for offices with
**more than 130** requests. This condition is on the aggregate result, so it
belongs in HAVING, not WHERE.

Try writing it with `WHERE COUNT(*) > 130` first if you want to see the
error — then fix it.

In [27]:
# Group requests by office
# Count the number of requests for each office
# HAVING filters the groups AFTER they have been counted.
busy_offices = pd.read_sql("""
    SELECT
        office_code,
        COUNT(*) AS request_count
    FROM service_requests
    GROUP BY office_code
    HAVING COUNT(*) > 130
    ORDER BY request_count DESC
""", conn)

# Display the results
busy_offices

,office_code,request_count
0,DICT-R6-01,218
1,DICT-R3-01,208
2,DICT-R11-01,198


### Task 2.4 — Why COUNT(*) and COUNT(column) disagree

Run the two counts below into `count_star` and `count_days`, then set
`count_diff` to their difference. This is the number of requests that have
no resolution time recorded yet — think about which statuses that must be.

In [28]:
# Count all requests, including rows where days_to_resolve is NULL
count_star = pd.read_sql("""
    SELECT COUNT(*) AS n
    FROM service_requests
""", conn)["n"].iloc[0]

# Count only requests that have a recorded resolution time
# COUNT(column) does NOT count NULL values.
count_days = pd.read_sql("""
    SELECT COUNT(days_to_resolve) AS n
    FROM service_requests
""", conn)["n"].iloc[0]

# Calculate how many requests have no resolution time recorded
count_diff = count_star - count_days

# Display the results
print("COUNT(*)        :", count_star)
print("COUNT(days_to_resolve):", count_days)
print("Difference      :", count_diff)

COUNT(*)        : 1222
COUNT(days_to_resolve): 824
Difference      : 398


#### Check Section 2

In [29]:
check_section_2()

Section 2 — Aggregation with GROUP BY and HAVING
  [PASS]  Task 1
  [PASS]  Task 2
  [PASS]  Task 3
  [PASS]  Task 4

  Section 2: 4 of 4 tasks passing.



np.True_

**Analysis and Discussion 2**

- One service type has the highest count but not the worst average
  resolution time. What might explain that?
- Why can `WHERE` not filter on `COUNT(*)`? Explain in terms of execution
  order.
- The average resolution days for some groups look pulled upward. What do
  you suspect, and how would you confirm it?

---
# Section 3 — Joins and validation

**Hands-On 3 · 60 minutes**

Bring in the offices data, join it to what you already have, and build the
habit that saves you from every silently-wrong join: count before, join,
count after, compare.

### Task 3.1 — Load and flatten the offices JSON

Load `data/regional_offices.json`, flatten it with `pd.json_normalize`, and
load it into a SQLite table named `offices`.

`services_offered` holds a Python list, and SQLite cannot store a list —
convert that column to text with `.astype(str)` before loading, or the
`.to_sql()` call will fail.

In [42]:
# Open the regional offices JSON file
with open("data/regional_offices.json", "r") as f:
    data = json.load(f)


# Get the list of offices from the JSON
# The actual office records are stored under the "offices" key
office_records = data["offices"]


# Flatten the nested JSON structure
# This will flatten location and coordinates into separate columns
offices = pd.json_normalize(office_records)


# Convert services_offered from a Python list to text
# SQLite cannot store Python lists directly
offices["services_offered"] = offices["services_offered"].astype(str)


# Load the flattened DataFrame into SQLite
# The table will be named "offices"
offices.to_sql("offices", conn, if_exists="replace", index=False)


# Display the loaded SQLite table
pd.read_sql("""
    SELECT *
    FROM offices
""", conn)

,office_code,office_name,staff_count,year_established,annual_budget_php,services_offered,is_active,location.region,location.city,location.coordinates.lat,location.coordinates.lon
0,DICT-NCR-01,DICT Field Office 01 - NCR,38,2019,2750000,"['eGov App Support', 'Business Permit Assistan...",1,NCR,Quezon City,7.1134,120.0971
1,DICT-NCR-02,DICT Field Office 02 - NCR,8,2012,6750000,"['Digital Literacy Training', 'Business Permit...",1,NCR,Quezon City,8.8445,123.0265
2,DICT-R3-01,DICT Field Office 01 - Region III,27,2021,2250000,"['Digital Literacy Training', 'eGov App Suppor...",1,Region III,San Fernando,9.6069,122.0509
3,DICT-R4A-01,DICT Field Office 01 - Region IV-A,33,2019,10500000,"['Business Permit Assistance', 'Digital Litera...",1,Region IV-A,Calamba,8.3352,124.1817
4,DICT-R4A-02,DICT Field Office 02 - Region IV-A,38,2015,10250000,"['ICT Equipment Request', 'Business Permit Ass...",1,Region IV-A,Calamba,9.5719,121.7123
5,DICT-R6-01,DICT Field Office 01 - Region VI,34,2015,3250000,"['Cybersecurity Advisory', 'Free WiFi Installa...",0,Region VI,Iloilo City,16.0067,123.9159
6,DICT-R7-01,DICT Field Office 01 - Region VII,16,2018,7500000,"['eGov App Support', 'Cybersecurity Advisory',...",1,Region VII,Cebu City,15.1750,123.7193
7,DICT-R7-02,DICT Field Office 02 - Region VII,33,2021,6250000,"['eGov App Support', 'Cybersecurity Advisory',...",1,Region VII,Cebu City,7.2545,123.2207
8,DICT-R11-01,DICT Field Office 01 - Region XI,21,2021,4250000,"['eGov App Support', 'Cybersecurity Advisory',...",1,Region XI,Davao City,11.7517,125.0756


### Task 3.2 — Join service requests to offices, and validate the row count

Set `joined` to an INNER JOIN of `service_requests` to `offices` on
`office_code`, returning `request_id`, `region`, `office_name` and
`staff_count`.

Then set `join_rows` to `len(joined)` and compare it to `total_rows` from
Section 1. They should match — print both and say out loud why.

In [44]:
# Join service_requests to offices using office_code
# INNER JOIN keeps only service requests that have a matching
# office_code in the offices table.
#
# Select only the columns requested by the task.
joined = pd.read_sql("""
    SELECT
        sr.request_id,
        sr.region,
        o.office_name,
        o.staff_count
    FROM service_requests AS sr
    INNER JOIN offices AS o
        ON sr.office_code = o.office_code
""", conn)


# Count the rows in the joined DataFrame
join_rows = len(joined)


# Compare the joined row count with total_rows
print("Total rows:", total_rows)
print("Joined rows:", join_rows)


# Verify that the row counts match
print("Do they match?", join_rows == total_rows)

Total rows: 1222
Joined rows: 1222
Do they match? True


### Task 3.3 — Workload per office

Set `workload` to, for each office: `office_name`, `staff_count`, a
`request_count`, and `requests_per_staff` (request_count divided by
staff_count, rounded to 2 decimals), ordered by `requests_per_staff`
descending.

SQLite does integer division by default — if every ratio comes back as
`0.0`, multiply the numerator by `1.0` (or `CAST` it) before dividing.

In [50]:
# Step 1: Calculate the number of service requests for each office
# COUNT(sr.request_id) counts how many requests belong to each office.
#
# Step 2: Calculate requests_per_staff
# Multiply request_count by 1.0 before dividing to force decimal division
# in SQLite.
#
# Step 3: ROUND(..., 2) rounds the ratio to 2 decimal places.
#
# Step 4: ORDER BY requests_per_staff DESC
# puts the offices with the highest workload per staff member first.

workload = pd.read_sql("""
    SELECT
        o.office_name,
        o.staff_count,
        COUNT(sr.request_id) AS request_count,
        ROUND(COUNT(sr.request_id) * 1.0 / o.staff_count, 2) AS requests_per_staff
    FROM offices AS o
    INNER JOIN service_requests AS sr
        ON o.office_code = sr.office_code
    GROUP BY
        o.office_code,
        o.office_name,
        o.staff_count
    ORDER BY requests_per_staff DESC
""", conn)


# Step 5: Display the workload table
workload


,office_name,staff_count,request_count,requests_per_staff
0,DICT Field Office 02 - NCR,8,98,12.25
1,DICT Field Office 01 - Region XI,21,198,9.43
2,DICT Field Office 01 - Region III,27,208,7.70
3,DICT Field Office 01 - Region VI,34,218,6.41
4,DICT Field Office 01 - Region VII,16,86,5.38
5,DICT Field Office 01 - Region IV-A,33,118,3.58
6,DICT Field Office 02 - Region VII,33,93,2.82
7,DICT Field Office 02 - Region IV-A,38,104,2.74
8,DICT Field Office 01 - NCR,38,99,2.61


#### Check Section 3

In [51]:
check_section_3()

Section 3 — Joins and validation
  [PASS]  Task 1
  [PASS]  Task 2
  [PASS]  Task 3
  [PASS]  Task 4

  Section 3: 4 of 4 tasks passing.



np.True_

**Analysis and Discussion 3**

- The row count did not change after the join. Why, and what would it have
  meant if it had grown?
- Which office has the heaviest workload per staff member? Is that a
  staffing problem or a demand problem?
- What would change if you used LEFT JOIN instead of INNER JOIN in Task
  3.2? Would any row appear or disappear here specifically?

### Bonus Challenge

Write a query that answers: “Which service types have the most Escalated requests in each region?” Use GROUP BY on two columns (region and service_type), and COUNT(*) filtered by `status = 'Escalated'`. Order the results in descending order by count.

In [52]:
# Step 1: Group the requests by region and service type
# This gives us one row for each region + service combination.
#
# Step 2: Count only requests where status is "Escalated"
# COUNT(*) counts the rows after the WHERE filter.
#
# Step 3: Order from highest to lowest number of escalated requests.

escalated_by_service = pd.read_sql("""
    SELECT
        region,
        service_type,
        COUNT(*) AS escalated_count
    FROM service_requests
    WHERE status = 'Escalated'
    GROUP BY
        region,
        service_type
    ORDER BY escalated_count DESC
""", conn)


# Step 4: Display the results
escalated_by_service

,region,service_type,escalated_count
0,Region III,eGov App Support,7
1,Region VI,Business Permit Assistance,7
2,Region VI,ICT Equipment Request,7
3,Region VII,ICT Equipment Request,7
4,Region IV-A,Cybersecurity Advisory,6
5,Region VI,eGov App Support,6
6,Region XI,Free WiFi Installation,6
7,Region III,ICT Equipment Request,5
8,Region IV-A,Business Permit Assistance,5
9,Region IV-A,Digital Literacy Training,5


---
# Final Check

Run this once all three sections are passing. It re-runs every task from the
top, so it also confirms nothing you changed later broke something earlier.

In [53]:
check_everything()

FINAL CHECK  —  all three sections

Section 1 — Relational thinking and basic queries
  [PASS]  Task 1
  [PASS]  Task 2
  [PASS]  Task 3
  [PASS]  Task 4

  Section 1: 4 of 4 tasks passing.

Section 2 — Aggregation with GROUP BY and HAVING
  [PASS]  Task 1
  [PASS]  Task 2
  [PASS]  Task 3
  [PASS]  Task 4

  Section 2: 4 of 4 tasks passing.

Section 3 — Joins and validation
  [PASS]  Task 1
  [PASS]  Task 2
  [PASS]  Task 3
  [PASS]  Task 4

  Section 3: 4 of 4 tasks passing.


   *  *  *   C O N G R A T U L A T I O N S   *  *  *

   All 12 tasks passed.

   You can now stand up a relational database inside a notebook, ask it
   direct questions with WHERE, GROUP BY and HAVING, and join two tables
   without silently duplicating a single row.

   More to the point, you caught a mis-typed category hiding inside
   'by_service', an office count that only appears once you group correctly,
   and a division bug that would have quietly shaved the decimals off every
   workload figure.

   

Before you close this notebook:

1. File > Download > .ipynb, and keep your copy.
2. One thing from today you would teach first, and why — write it in the
   cell below.
3. Tomorrow (Day 4) picks up exactly where Task 3.1 leaves off: importing
   and flattening data from multiple sources. The JSON-flattening habit you
   just built is not a one-time trick.

*Your answer:*

